In [15]:
import os
import numpy as np
import easyvvuq as uq
import chaospy as cp
import matplotlib.pyplot as plt
from easyvvuq.actions import CreateRunDirectory, Encode, Decode, ExecuteLocal, Actions
from util import persist

# EasyVVUQ on many Benchmarks

## Setting up an EasyVVUQ campaign

In [16]:
# Quantity of Interest
QOI = 'energy_uj'

# location where the run directories are stored
WORK_DIR = '.results'

We first set up the `params` dictionary, in which we specify the name, type and default value of each input
Also the `vary` dictionary, which holds the `chaospy` distribution of each input

In [17]:
params = {}
vary = {}

# Current machine maximum number of cores
params['N_THREADS'] = {'type': 'integer', 'default': 16}
vary['N_THREADS'] = cp.DiscreteUniform(1, 16)

# Levels of Clock speed, for our current machine:
# 2200000 = 0,
# 2800000 = 1,
# 3300000 = 2
params['CLK'] = {'type': 'integer', 'default': 2}
vary['CLK'] = cp.DiscreteUniform(0, 2)

# params['POWER_CAP'] = {'type': 'integer', 'default': 220.0}  # power cap in watts

d = len(params)

In [18]:
# input file encoder
encoder = uq.encoders.GenericEncoder(template_fname='energy.template', delimiter='$', target_filename='input.csv')

The wrapper writes a CSV file `output.csv` containing the energy, in microjoules, used during the programs execution.

In [19]:
# CSV output file decoder
decoder = uq.decoders.SimpleCSV(target_filename='output.csv', output_columns=[QOI])

In [20]:
# Local execution of the wrapper around benchmarks
execute = ExecuteLocal(f'{os.getcwd()}/energy_wrapper.py')

Now we are combine all actions we want to execute into an `Actions` object.

In [21]:
# actions to be undertaken: make rundirs, encode input files, execute local model ensemble, decode output files
actions = Actions(
    CreateRunDirectory(root=WORK_DIR, flatten=True),
    Encode(encoder),
    execute,
    Decode(decoder)
)

The central object in the UQ analysis is a so-called Campaign. This is created as:

In [22]:
campaign = uq.Campaign(name='energy', params=params, actions=actions, work_dir=WORK_DIR)

We now select the adaptive Stochastic Collocation sampler. Here

* `polynomial_order = 1`: should be interpreted in the sparse context as starting the sampling plan with a level 1 quadrature rule for all inputs.
* `quadrature_rule='C'`: selects the Clenshaw Curtis quadrature rule.
* `sparse=True`: selects the sparse grid.
* `growth=True`: selects an exponential growth rule which makes the Clenshaw Curtis rule nested.
* `dimension_adaptive=True`: selects the dimension-adaptive sampler.

In [23]:
sampler = uq.sampling.SCSampler(vary=vary, polynomial_order=1, quadrature_rule='C', sparse=True, midpoint_level1=True, dimension_adaptive=True)
campaign.set_sampler(sampler)

## Run campaign and adaptation

Run the first ensemble, which consists of just a single sample:

In [24]:
campaign.execute(sequential=True).collate(progress_bar=True)

Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 49204685040

Running NONE with 8 threads

energy_uj: Stdout: 49205238738
NONE
    Stdout
        Running NONE with 8 threads
        OMP_NUM_THREADS=8
        OMP_PROC_BIND=CLOSE
        OMP_PLACES=CORES
        "Running None"


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 327.68it/s]


To analyse the results (and execute the dimension adaptivity), we need an `SCAnalysis` object:

In [25]:
analysis = uq.analysis.SCAnalysis(sampler=sampler, qoi_cols=[QOI])

In [26]:
# perform analysis (basically estimates moments, Sobol analysis, and updates internal state of analysis)
campaign.apply_analysis(analysis)

Now we'll refine the grid several times in an anisotropic fashion. Here

* `look_ahead`: determines the new admissible candidate refinements.
* `campaign.get_collation_result()`: get the data frame with all code samples.
* `adapt_dimension`: compute the hierarchical surplus at all candidate refinements, and accept the one with the highest surplus.

In [27]:
def plot_new_points(new_points):
      plt.figure()
      xs = np.array([x for x, y in new_points])
      ys = np.array([y for x, y in new_points])
      plt.plot(xs, ys, 'o')
      plt.show()

def refine_sampling_plan(number_of_refinements):
      """
      Refine the sampling plan.

      Parameters
      ----------
      number_of_refinements (int)
      The number of refinement iterations that must be performed.

      Returns
      -------
      None. The new accepted indices are stored in analysis.l_norm and the admissible indices
      in sampler.admissible_idx.
      """
      for i in range(number_of_refinements):
      # compute the admissible indices
            sampler.look_ahead(analysis.l_norm)

            if sampler.n_new_points[-1] == 0:
                  # maybe we should stop??
                  pass

            if len(sampler.admissible_idx) == 0:
                  # we searched everything
                  break
            
            print(f"-----------------------{i + 1}------------------------")
            print(f"-----------------------{sampler.n_new_points[-1]} new points------------------------")
            # run the ensemble
            campaign.execute(sequential=True).collate(progress_bar=True)

            # accept one of the multi indices of the new admissible set
            data_frame = campaign.get_collation_result()
            analysis.adapt_dimension(QOI, data_frame, method='var')


In [ ]:
refine_sampling_plan(50)

-----------------------1------------------------
-----------------------4 new points------------------------


Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 47986778038

Running NONE with 8 threads

energy_uj: Stdout: 47987596004
NONE
    Stdout
        Running NONE with 8 threads
        OMP_NUM_THREADS=8
        OMP_PROC_BIND=CLOSE
        OMP_PLACES=CORES
        "Running None"
Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 48006434049

Running NONE with 12 threads

energy_uj: Stdout: 48006943117
NONE
    Stdout
        Running NONE with 12 threads
        OMP_NUM_THREADS=12
        OMP_PROC_BIND=CLOSE
        OMP_PLACES=CORES
        "Running None"
Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 48017507390

Running NONE with 5 threads

energy_uj: Stdout: 48018002696
NONE
    Stdout
        Running NONE with 5 threads
        OMP_NUM_THREADS=5
        OMP_PROC_BIND=CLOSE
        OMP_PLACES=CORES
        "Running None"
Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 48028341425


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 660.91it/s]


-----------------------2------------------------
-----------------------0 new points------------------------


0it [00:00, ?it/s]


...

-----------------------13------------------------
-----------------------0 new points------------------------


0it [00:00, ?it/s]


-----------------------14------------------------
-----------------------10 new points------------------------
Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 48926586263

Running NONE with 6 threads

energy_uj: Stdout: 48927370158
NONE
    Stdout
        Running NONE with 6 threads
        OMP_NUM_THREADS=6
        OMP_PROC_BIND=CLOSE
        OMP_PLACES=CORES
        "Running None"
Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 48946568094

Running NONE with 4 threads

energy_uj: Stdout: 48947206076
NONE
    Stdout
        Running NONE with 4 threads
        OMP_NUM_THREADS=4
        OMP_PROC_BIND=CLOSE
        OMP_PLACES=CORES
        "Running None"
Running on Glados
Governor set: Done
Frequency set: Done
energy_uj: Stdout: 48957921144

Running NONE with 9 threads

energy_uj: Stdout: 48958530884
NONE
    Stdout
        Running NONE with 9 threads
        OMP_NUM_THREADS=9
        OMP_PROC_BIND=CLOSE
        OMP_PLACES=CORES
   

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 2015.04it/s]


KeyboardInterrupt: 

In [ ]:
len(analysis.l_norm)

21

In [ ]:
campaign.apply_analysis(analysis)
data_frame = campaign.get_collation_result()

In [ ]:
RESULTS_DIR = "run_results/"
persist.save(analysis, sampler, data_frame, vary, [QOI], RESULTS_DIR)